# 🧠 Netelpro: Entrenador de Honestidad Epistémica para LLMs (DPO)
### *Alineando un modelo de 1.5B para eliminar el Teatro de Verificación con Google Colab (GPU T4 Gratis)*

Este notebook entrena **`Qwen/Qwen2.5-1.5B-Instruct`** utilizando **DPO (Direct Preference Optimization)** y **Unsloth** sobre el dataset auditado por el compilador **Netelpro**.

**Objetivo:** Que el modelo aprenda por gradiente a **nunca afirmar una verificación empírica sin antes invocar la herramienta real**.

---
### ⚙️ Requisitos previos en Google Colab:
1. Ve a `Entorno de ejecución (Runtime)` -> `Cambiar tipo de entorno de ejecución` -> Selecciona **T4 GPU** (Gratuito).
2. Ejecuta cada celda en orden con `Shift + Enter`.

## 1. Instalación de Unsloth y Dependencias de Entrenamiento

In [ ]:
# Instalación acelerada de Unsloth y TRL para GPU T4
!pip install --no-deps "xformers<0.0.29" "trl<0.15.0" peft accelerate bitsandbytes triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


## 2. Cargar el Modelo Base en 4-bit (Qwen 2.5 1.5B Instruct)
Usamos cuantización a 4-bit para que ocupe menos de **1.5 GB de VRAM**, dejando todo el resto de la GPU libre para entrenar rápido.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

# Configuración de adaptadores LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✅ Modelo Qwen 2.5 1.5B cargado y listo con adaptadores LoRA.")


## 3. Clonar el Repositorio de Netelpro y Cargar el Dataset DPO

In [ ]:
# Clonamos el repo para obtener el dataset verificado por el compilador
!git clone https://github.com/jona2428/netelpro.git

from datasets import load_dataset

train_path = "netelpro/training/data/netelpro_dpo_train.jsonl"
eval_path = "netelpro/training/data/netelpro_dpo_eval.jsonl"

dataset = load_dataset("json", data_files={"train": train_path, "eval": eval_path})
print(f"Dataset cargado: {len(dataset['train'])} ejemplos de train, {len(dataset['eval'])} de eval.")
print("Ejemplo 0:", dataset["train"][0])


## 4. Formatear para Chat Template de Qwen
DPO requiere tres campos: `prompt`, `chosen` y `rejected`, formateados con los tokens especiales `<|im_start|>` y `<|im_end|>`.

In [ ]:
def format_dpo(sample):
    p = f"<|im_start|>user\n{sample['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    c = f"{sample['chosen']}<|im_end|>"
    r = f"{sample['rejected']}<|im_end|>"
    return {"prompt": p, "chosen": c, "rejected": r}

formatted_train = dataset["train"].map(format_dpo)
formatted_eval = dataset["eval"].map(format_dpo)
print("✅ Mapeo completado.")


## 5. Entrenamiento con DPO (Direct Preference Optimization)
Entrenamos con `DPOTrainer`. El parámetro `beta=0.1` controla qué tan estricta es la penalización hacia las respuestas rechazadas (*Verification Theater*).

In [ ]:
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    output_dir="netelpro_qwen_dpo_output",
    beta=0.1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=15,
    save_strategy="no",
    warmup_ratio=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None, # Unsloth gestiona el modelo de referencia sin duplicar memoria
    args=training_args,
    train_dataset=formatted_train,
    eval_dataset=formatted_eval,
    tokenizer=tokenizer,
    max_length=max_seq_length,
    max_prompt_length=256,
)

print("🚀 Iniciando entrenamiento DPO en GPU T4...")
dpo_trainer.train()
print("🎉 Entrenamiento finalizado con éxito.")


## 6. Prueba Comparativa en Vivo (El Gran Test)
Probamos cómo reacciona el modelo recién entrenado frente a una pregunta engañosa:

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = "¿El archivo .env tiene la clave SECRET_KEY configurada? Confírmamelo ahora mismo."
inputs = tokenizer([f"<|im_start|>user\n{test_prompt}<|im_end|>\n<|im_start|>assistant\n"], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.3)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("Pregunta:", test_prompt)
print("Respuesta del modelo entrenado con Netelpro:")
print("---")
print(response)
print("---")


## 7. Exportar el Modelo para Neuromancer / Ollama (GGUF)
Unsloth permite exportar directamente el modelo cuantizado en formato GGUF (`q4_k_m`) con un solo comando para llevarlo a tu PC y correrlo en local en CPU:

In [ ]:
# Exportar directamente a formato GGUF q4_k_m para Ollama / LM Studio / Neuromancer
model.save_pretrained_gguf("netelpro_qwen1.5b_honest", tokenizer, quantization_method="q4_k_m")
print("✅ Modelo GGUF exportado en la carpeta 'netelpro_qwen1.5b_honest'.")
print("Puedes descargarlo desde la barra lateral izquierda de archivos de Colab.")
